# Plant Pathology 2021 — Ensemble Inference (EfficientNet-B0 + ConvNeXt-Tiny + Swin-T)

This notebook **does not train anything**. It loads three checkpoints trained separately
(each with its own recipe — Swin runs at 224px, the CNNs at 256px), averages their
sigmoid probabilities with TTA, tunes a single set of per-class thresholds on the
shared validation split, and writes `submission.csv`.

- **Metric:** Mean F1-Score
- **Runtime:** Kaggle GPU T4, internet disabled
- **Ensembling happens on probabilities** (after sigmoid), not on labels or logits.

Required Add Data:
1. the checkpoint dataset (three `.pt` files);
2. the 512px train dataset (only for re-tuning ensemble thresholds on validation);
3. the competition data (hidden test is substituted here at submission).

In [ ]:
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from PIL import Image
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
from torch.amp import autocast
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
from tqdm.auto import tqdm

In [ ]:
class CFG:
    # Checkpoints trained by the training notebook (one .pt per backbone).
    CKPT_DIR = Path("/kaggle/input/datasets/sergeisemenovdev/models-apples-tree-deseases")
    CKPT_FILES = [
        "best_model_efficientnet_b0_v2.pt",
        "best_model_convnext_tiny_v2.pt",
        "best_model_swin_t_v2.pt",
    ]

    # 512px train data — needed ONLY to rebuild the same validation split and
    # re-tune ensemble thresholds on the averaged probabilities.
    DATA_DIR = Path("/kaggle/input/datasets/sergeisemenovdev/plant-pathology-512/plant-pathology-512")
    TRAIN_CSV = DATA_DIR / "train.csv"
    TRAIN_IMG_DIR = DATA_DIR / "train_images"

    # Official competition dataset — the hidden test is substituted HERE at submission.
    COMP_DIR = Path("/kaggle/input/competitions/plant-pathology-2021-fgvc8")
    TEST_IMG_DIR = COMP_DIR / "test_images"

    OUTPUT_DIR = Path("/kaggle/working")

    BATCH_SIZE = 64
    NUM_WORKERS = 0    # inference: single-process (draft decode is cheap). Building and
                       # freeing several models in a loop with worker processes spews
                       # harmless "can only test a child process" shutdown tracebacks;
                       # 0 workers avoids that noise and doesn't change results.
    VAL_RATIO = 0.15   # must match the training notebook to reproduce the split
    SEED = 42          # must match the training notebook
    TTA = True         # original + hflip + vflip, averaged


def seed_everything(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


seed_everything(CFG.SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

In [ ]:
# Labels: rebuild exactly as in training (sorted -> stable index order).
train_df = pd.read_csv(CFG.TRAIN_CSV)
train_df["label_list"] = train_df["labels"].apply(lambda s: s.strip().split())

all_labels = sorted({l for ls in train_df["label_list"] for l in ls})
label2idx = {l: i for i, l in enumerate(all_labels)}
idx2label = {i: l for l, i in label2idx.items()}
num_classes = len(all_labels)
print(f"Classes ({num_classes}): {all_labels}")


def labels_to_multihot(label_list, n):
    v = np.zeros(n, dtype=np.float32)
    for l in label_list:
        v[label2idx[l]] = 1.0
    return v


train_df["target"] = train_df["label_list"].apply(lambda x: labels_to_multihot(x, num_classes))

In [ ]:
# Reproduce the SAME stratified validation split used in training (same seed/ratio),
# so ensemble thresholds are tuned on data none of the models trained on.
label_counts = train_df["labels"].value_counts()
stratifiable_mask = train_df["labels"].map(label_counts) >= 2
strat_indices = np.where(stratifiable_mask)[0]

_, val_idx = train_test_split(
    strat_indices,
    test_size=CFG.VAL_RATIO,
    random_state=CFG.SEED,
    shuffle=True,
    stratify=train_df["labels"].iloc[strat_indices],
)
val_split = train_df.iloc[val_idx].reset_index(drop=True)
print(f"Validation samples: {len(val_split)}")

In [ ]:
# --- Model definition (identical architecture to the training notebook) ---
def load_backbone(backbone_name: str):
    """Build the architecture WITHOUT ImageNet weights — we load our trained
    checkpoint into the full model afterwards."""
    if backbone_name == "efficientnet_b0":
        base = models.efficientnet_b0(weights=None)
        in_features = base.classifier[1].in_features
        base.classifier = nn.Identity()
        return base, in_features
    if backbone_name == "resnet50":
        base = models.resnet50(weights=None)
        in_features = base.fc.in_features
        base.fc = nn.Identity()
        return base, in_features
    if backbone_name == "convnext_tiny":
        base = models.convnext_tiny(weights=None)
        in_features = base.classifier[2].in_features
        base.classifier[2] = nn.Identity()
        return base, in_features
    if backbone_name == "swin_t":
        base = models.swin_t(weights=None)
        in_features = base.head.in_features
        base.head = nn.Identity()
        return base, in_features
    raise ValueError(f"Unknown backbone: {backbone_name}")


class MultiLabelCNN(nn.Module):
    def __init__(self, backbone_name: str, num_classes: int):
        super().__init__()
        self.backbone, in_features = load_backbone(backbone_name)
        self.head = nn.Sequential(nn.Dropout(p=0.3), nn.Linear(in_features, num_classes))

    def forward(self, x):
        return self.head(self.backbone(x))

In [ ]:
# --- Data plumbing (per-model IMG_SIZE, so we build loaders on demand) ---
def load_image_fast(path: Path, size: int) -> Image.Image:
    image = Image.open(path)
    image.draft("RGB", (size, size))
    return image.convert("RGB")


def make_transform(img_size: int):
    return transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])


class ValDataset(Dataset):
    def __init__(self, df, img_dir, img_size):
        self.image_names = df["image"].to_numpy()
        self.img_dir = img_dir
        self.tf = make_transform(img_size)
        self.img_size = img_size

    def __len__(self):
        return len(self.image_names)

    def __getitem__(self, idx):
        img = load_image_fast(self.img_dir / self.image_names[idx], self.img_size)
        return self.tf(img), idx


class TestDataset(Dataset):
    def __init__(self, img_dir, img_size):
        self.img_dir = img_dir
        self.image_names = sorted(
            p.name for p in img_dir.iterdir() if p.suffix.lower() in {".jpg", ".jpeg", ".png"}
        )
        self.tf = make_transform(img_size)
        self.img_size = img_size

    def __len__(self):
        return len(self.image_names)

    def __getitem__(self, idx):
        img = load_image_fast(self.img_dir / self.image_names[idx], self.img_size)
        return self.tf(img), idx


def make_loader(dataset):
    return DataLoader(
        dataset,
        batch_size=CFG.BATCH_SIZE,
        shuffle=False,
        num_workers=CFG.NUM_WORKERS,
        pin_memory=True,
    )

In [ ]:
@torch.no_grad()
def predict_probs(model, loader, tta: bool = True) -> np.ndarray:
    """Return (N, num_classes) averaged sigmoid probabilities in dataset order.
    TTA: original + horizontal flip + vertical flip, averaged."""
    model.eval()
    n = len(loader.dataset)
    out = np.zeros((n, num_classes), dtype=np.float64)

    for images, idxs in tqdm(loader, leave=False):
        images = images.to(device, non_blocking=True)
        views = [images]
        if tta:
            views.append(torch.flip(images, dims=[3]))  # hflip
            views.append(torch.flip(images, dims=[2]))  # vflip
        probs_sum = None
        for v in views:
            with autocast("cuda"):
                logits = model(v)
            p = torch.sigmoid(logits).float()
            probs_sum = p if probs_sum is None else probs_sum + p
        probs = (probs_sum / len(views)).cpu().numpy()
        out[idxs.numpy()] = probs
    return out

In [ ]:
# --- Run every model once: collect its VAL and TEST probabilities, then average ---
# Each checkpoint carries its own backbone name and IMG_SIZE in ckpt['cfg'],
# so Swin is evaluated at 224 and the CNNs at 256 automatically.
val_prob_sum = np.zeros((len(val_split), num_classes), dtype=np.float64)
test_prob_sum = None
test_names = None
n_models = 0

for fname in CFG.CKPT_FILES:
    ckpt = torch.load(CFG.CKPT_DIR / fname, map_location="cpu", weights_only=False)
    backbone = ckpt["cfg"]["BACKBONE"]
    img_size = int(ckpt["cfg"]["IMG_SIZE"])
    print(f"\n{fname}: backbone={backbone}, img_size={img_size}, val_f1={ckpt.get('best_f1'):.4f}")

    model = MultiLabelCNN(backbone, num_classes).to(device)
    model.load_state_dict(ckpt["model_state_dict"])

    # validation probabilities (for threshold tuning)
    val_loader = make_loader(ValDataset(val_split, CFG.TRAIN_IMG_DIR, img_size))
    val_prob_sum += predict_probs(model, val_loader, tta=CFG.TTA)

    # test probabilities
    test_ds = TestDataset(CFG.TEST_IMG_DIR, img_size)
    test_loader = make_loader(test_ds)
    tp = predict_probs(model, test_loader, tta=CFG.TTA)
    test_prob_sum = tp if test_prob_sum is None else test_prob_sum + tp
    if test_names is None:
        test_names = list(test_ds.image_names)

    n_models += 1
    del model
    torch.cuda.empty_cache()

val_probs = val_prob_sum / n_models
test_probs = test_prob_sum / n_models
print(f"\nEnsembled {n_models} models. Test images: {len(test_names)}")

In [ ]:
# --- Tune per-class thresholds on the ENSEMBLE's averaged validation probabilities ---
def competition_f1(y_true, y_pred):
    return float(np.mean([f1_score(yt, yp, average="macro", zero_division=0)
                          for yt, yp in zip(y_true, y_pred)]))


val_targets = np.stack(val_split["target"].to_numpy())

thresholds = np.full(num_classes, 0.5)
base_f1 = competition_f1(val_targets, (val_probs >= thresholds).astype(int))
print(f"Ensemble val F1 @ flat 0.5   : {base_f1:.4f}")

for c in range(num_classes):
    best_t, best_score = thresholds[c], -1.0
    for t in np.arange(0.20, 0.71, 0.05):
        trial = thresholds.copy()
        trial[c] = t
        s = competition_f1(val_targets, (val_probs >= trial).astype(int))
        if s > best_score:
            best_score, best_t = s, t
    thresholds[c] = best_t

tuned_f1 = competition_f1(val_targets, (val_probs >= thresholds).astype(int))
print(f"Ensemble val F1 @ per-class  : {tuned_f1:.4f}")
for l, i in label2idx.items():
    print(f"  {l:>22}: {thresholds[i]:.2f}")

In [ ]:
# --- Build submission from averaged test probabilities + tuned thresholds ---
def probs_to_label_string(probs, thresholds):
    idx = np.where(probs >= thresholds)[0]
    if len(idx) == 0:
        idx = [int(probs.argmax())]  # never emit an empty label
    return " ".join(idx2label[i] for i in idx)


label_strings = [probs_to_label_string(p, thresholds) for p in test_probs]
submission = pd.DataFrame({"image": test_names, "labels": label_strings})
submission = submission.sort_values("image").reset_index(drop=True)

sub_path = CFG.OUTPUT_DIR / "submission.csv"
submission.to_csv(sub_path, index=False)
print(f"Submission saved: {sub_path}  ({len(submission)} rows)")
submission.head(10)